In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import mlflow
import os
from kaggle_secrets import UserSecretsClient

INPUT_DIR = "<link-to-the-2nd-notebook>"
features_df = pd.read_parquet(f"{INPUT_DIR}/retailcast_features.parquet")
features_df["date"] = pd.to_datetime(features_df["date"])

In [ ]:
secrets = UserSecretsClient()
DAGSHUB_TOKEN = secrets.get_secret("DAGSHUB_TOKEN")
DAGSHUB_REPO = secrets.get_secret("DAGSHUB_REPO")  # e.g. "yourusername/retailcast"

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_TOKEN
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_REPO}.mlflow")
mlflow.set_experiment("retailcast_ml_models")

In [ ]:
HORIZON = 15
N_FOLDS = 4

def get_walkforward_folds(all_dates, horizon=HORIZON, n_folds=N_FOLDS):
    all_dates = sorted(all_dates)
    last_date = all_dates[-1]
    holdout_start = last_date - pd.Timedelta(days=horizon - 1)
    folds = []
    for i in range(n_folds, 0, -1):
        test_end = holdout_start - pd.Timedelta(days=(i - 1) * horizon + 1)
        test_start = test_end - pd.Timedelta(days=horizon - 1)
        train_end = test_start - pd.Timedelta(days=1)
        folds.append((train_end, test_start, test_end))
    holdout = (holdout_start - pd.Timedelta(days=1), holdout_start, last_date)
    return folds, holdout

all_dates = features_df["date"].unique()
folds, holdout = get_walkforward_folds(all_dates)

def mape(actual, forecast):
    actual, forecast = np.array(actual), np.array(forecast)
    mask = actual != 0
    return np.mean(np.abs((actual[mask] - forecast[mask]) / actual[mask])) * 100

def wape(actual, forecast):
    actual, forecast = np.array(actual), np.array(forecast)
    return np.sum(np.abs(actual - forecast)) / np.sum(np.abs(actual)) * 100

def mase(actual, forecast, train_series, seasonal_period=7):
    actual, forecast = np.array(actual), np.array(forecast)
    train_series = np.array(train_series)
    naive_errors = np.abs(train_series[seasonal_period:] - train_series[:-seasonal_period])
    scale = np.mean(naive_errors) if len(naive_errors) > 0 else 1.0
    return np.mean(np.abs(actual - forecast)) / scale if scale != 0 else np.nan

In [ ]:
FEATURE_COLS = [
    "store_nbr", "family", "city", "state", "type", "cluster",
    "onpromotion", "is_holiday", "day_of_week", "month", "is_weekend",
    "day_of_month", "week_of_year", "dcoilwtico",
    "lag_7", "lag_14", "lag_28",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_28",
    "rolling_std_7", "rolling_std_14", "rolling_std_28",
]
CATEGORICAL_COLS = ["store_nbr", "family", "city", "state", "type", "cluster"]
TARGET = "sales"

for col in CATEGORICAL_COLS:
    features_df[col] = features_df[col].astype("category")

In [ ]:
def run_fold(model_type, train_df, test_df):
    X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET]
    X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET]

    if model_type == "lightgbm":
        model = lgb.LGBMRegressor(
            n_estimators=500, learning_rate=0.05, num_leaves=31,
            categorical_feature=CATEGORICAL_COLS, verbose=-1,
        )
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
    elif model_type == "xgboost":
        X_train_enc = X_train.copy()
        X_test_enc = X_test.copy()
        model = xgb.XGBRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            enable_categorical=True, tree_method="hist",
        )
        model.fit(X_train_enc, y_train)
        pred = model.predict(X_test_enc)

    pred = np.clip(pred, 0, None)
    return model, pred, y_test.values

ml_results = []
feature_importances = {"lightgbm": [], "xgboost": []}
final_holdout_preds = {}

for model_type in ["lightgbm", "xgboost"]:
    with mlflow.start_run(run_name=f"{model_type}_global_walkforward"):
        mlflow.log_param("model_type", model_type)
        mlflow.log_param("horizon", HORIZON)
        mlflow.log_param("n_folds", N_FOLDS)

        for fold_idx, (train_end, test_start, test_end) in enumerate(folds + [holdout]):
            fold_label = f"fold_{fold_idx+1}" if fold_idx < len(folds) else "holdout"

            train_df = features_df[features_df["date"] <= train_end]
            test_df = features_df[
                (features_df["date"] >= test_start) & (features_df["date"] <= test_end)
            ]
            if len(train_df) == 0 or len(test_df) == 0:
                continue

            model, pred, actual = run_fold(model_type, train_df, test_df)

            fold_mape = mape(actual, pred)
            fold_wape = wape(actual, pred)
            fold_mase = mase(actual, pred, train_df[TARGET].values)

            ml_results.append({
                "model": model_type, "fold": fold_label,
                "mape": fold_mape, "wape": fold_wape, "mase": fold_mase,
            })
            mlflow.log_metric(f"{fold_label}_mape", fold_mape)
            mlflow.log_metric(f"{fold_label}_wape", fold_wape)
            mlflow.log_metric(f"{fold_label}_mase", fold_mase)

            if fold_label == "holdout":
                test_df_result = test_df[["date", "store_nbr", "family", TARGET]].copy()
                test_df_result["forecast"] = pred
                final_holdout_preds[model_type] = test_df_result

                if model_type == "lightgbm":
                    importances = pd.DataFrame({
                        "feature": FEATURE_COLS, "importance": model.feature_importances_,
                    }).sort_values("importance", ascending=False)
                    importances.to_csv(f"/kaggle/working/{model_type}_feature_importance.csv", index=False)
                    mlflow.log_artifact(f"/kaggle/working/{model_type}_feature_importance.csv")

        mlflow.end_run()

ml_results_df = pd.DataFrame(ml_results)
print(ml_results_df.groupby(["model", "fold"])[["mape", "wape", "mase"]].mean())

In [ ]:
FAMILY_COST_PER_UNIT_ERROR = {
    "GROCERY I": 0.75, "BEVERAGES": 0.75, "CLEANING": 0.75, "HOME CARE": 0.75,
    "PRODUCE": 0.50, "DAIRY": 0.50,
}

for model_type, df in final_holdout_preds.items():
    df = df.copy()
    df["cost_per_unit"] = df["family"].map(FAMILY_COST_PER_UNIT_ERROR)
    df["abs_error"] = np.abs(df["sales"] - df["forecast"])
    estimated_cost = (df["abs_error"] * df["cost_per_unit"]).sum()
    total_abs_error = df["abs_error"].sum()
    print(f"{model_type}: total abs error units = {total_abs_error:.0f}, estimated cost = ${estimated_cost:,.2f}")

In [ ]:
ml_results_df.to_csv("/kaggle/working/ml_results.csv", index=False)

holdout_comparison = ml_results_df[ml_results_df["fold"] == "holdout"].sort_values("mase")
best_model = holdout_comparison.iloc[0]["model"]
print(f"Best model on holdout (by MASE): {best_model}")

final_holdout_preds[best_model].to_parquet("/kaggle/working/final_holdout_predictions.parquet", index=False)
print(f"Saved: ml_results.csv, final_holdout_predictions.parquet (using {best_model} predictions)")